In [1]:
conda install pandas numpy scipy openpyxl

Retrieving notices: ...working... done
Channels:
 - defaults
Platform: win-64
Solving environment: ...working... done

## Package Plan ##

  environment location: C:\Users\hp\anaconda3

  added / updated specs:
    - numpy
    - openpyxl
    - pandas
    - scipy


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.10.20 |       haa95532_0         113 KB
    certifi-2026.10.23         |     pyhd3eb1b0_1         131 KB
    openpyxl-3.1.5             |  py314h1c6eee0_2         617 KB
    pandas-3.0.6               |  py314hf1216d0_2        12.2 MB
    scipy-1.18.1               |  py314h176032c_0        21.0 MB
    ------------------------------------------------------------
                                           Total:        34.1 MB

The following packages will be UPDATED:

  ca-certificates                      2026.9.25-haa95532_0 --> 2026.10.20-haa95532_0 
  certi

Error while loading conda entry point: conda-anaconda-telemetry (cannot import name 'CondaRequestHeader' from 'conda.plugins' (C:\Users\hp\anaconda3\Lib\site-packages\conda\plugins\__init__.py))
Error while loading conda entry point: conda-anaconda-tos (cannot import name 'CondaPreCommand' from 'conda.plugins' (C:\Users\hp\anaconda3\Lib\site-packages\conda\plugins\__init__.py))
Error while loading conda entry point: repo (cannot import name 'CondaSubcommand' from 'conda.plugins' (C:\Users\hp\anaconda3\Lib\site-packages\conda\plugins\__init__.py))
Error while loading conda entry point: conda-anaconda-telemetry (cannot import name 'CondaRequestHeader' from 'conda.plugins' (C:\Users\hp\anaconda3\Lib\site-packages\conda\plugins\__init__.py))
Error while loading conda entry point: conda-anaconda-tos (cannot import name 'CondaPreCommand' from 'conda.plugins' (C:\Users\hp\anaconda3\Lib\site-packages\conda\plugins\__init__.py))
Error while loading conda entry point: repo (cannot import name 'C

In [15]:
#Uploadfile
import pandas as pd

file_path = "RQ3_human_evaluation_merged3.xlsx"

df = pd.read_excel(file_path, sheet_name="Combined")

In [14]:
#Wilcoxon p-value
from scipy.stats import wilcoxon
import pandas as pd

criteria = {
    "Relevance": "Mean_Relevance",
    "Correctness": "Mean_Correctness",
    "Faithfulness": "Mean_Faithfulness"
}

for criterion, column in criteria.items():

    paired = df.pivot(
        index="query_id",
        columns="system_label",
        values=column
    )

    A = paired["System A"].to_numpy()
    B = paired["System B"].to_numpy()

    result = wilcoxon(
        B, A,
        alternative="two-sided"
    )

    print(f"{criterion}: p = {result.pvalue:.4f}")

Relevance: p = 0.6318
Correctness: p = 0.4088
Faithfulness: p = 0.2906


In [12]:
# 95% Bootstrap Confidence Interval
import numpy as np

rng = np.random.default_rng(42)
n_bootstrap = 20000

for criterion, column in criteria.items():

    paired = df.pivot(
        index="query_id",
        columns="system_label",
        values=column
    )

    A = paired["System A"].to_numpy(dtype=float)
    B = paired["System B"].to_numpy(dtype=float)

    # الفرق لكل سؤال: Reranked - Non-reranked
    differences = B - A

    # متوسط الفرق الفعلي
    mean_diff = np.mean(differences)

    # Bootstrap: إعادة السحب مع الإرجاع
    boot_means = np.empty(n_bootstrap)

    for i in range(n_bootstrap):
        sample = rng.choice(
            differences,
            size=len(differences),
            replace=True
        )
        boot_means[i] = np.mean(sample)

    # فترة الثقة 95%
    lower, upper = np.percentile(
        boot_means, [2.5, 97.5]
    )

    print(f"\n{criterion}")
    print(f"Mean difference (B-A): {mean_diff:.4f}")
    print(f"95% Bootstrap CI: [{lower:.4f}, {upper:.4f}]")


Relevance
Mean difference (B-A): 0.0800
95% Bootstrap CI: [-0.1000, 0.3000]

Correctness
Mean difference (B-A): 0.1200
95% Bootstrap CI: [-0.0800, 0.3600]

Faithfulness
Mean difference (B-A): 0.1400
95% Bootstrap CI: [-0.0600, 0.3800]
